# **ARGUS-FEDER — Pelican Data Access, Metadata, and PTDATA Basics**

This notebook demonstrates ARGUS answering natural-language questions about
DIII-D data through several different access paths: semantic concept →
MDSplus signal resolution, the shot-metadata database, PTDATA (a separate
raw-digitizer data source), and IMAS (the device-neutral naming standard).
Each `%%ask` cell sends a plain-English request; ARGUS consults the DIII-D
skills, writes and runs the code, and reports back.

Run Steps 1–4 below once, then run any `%%ask` cell in any order.

#### Step 1 — Prepare your session (one-time)

Run the cell below. **Your session will restart automatically — that's
expected.** Once it restarts, continue to Step 2.

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

#### Step 2 — Choose your AI model

Pick one and run the cell (defaults to a free NRP model).

In [ ]:
# NRP (National Research Platform) -- default, free for NRP users
LLM = {
    "model": "glm-5",
    "url": "https://ellm.nrp-nautilus.io/v1",
    "api_key_env": "NRP_API_KEY",
}

# Anthropic Claude
# LLM = {
#     "model": "claude-sonnet-4-6",
#     "api_key_env": "ANTHROPIC_API_KEY",
#     "flavor": "anthropic",
# }

# OpenAI
# LLM = {
#     "model": "gpt-4o-mini",
#     "api_key_env": "OPENAI_API_KEY",
# }

#### Step 3 — Add your keys

Click the 🔑 icon in the left sidebar and add two secrets (toggle
"Notebook access" on for each):

- The API key matching what you picked in Step 2 (e.g. `NRP_API_KEY`)
- Your DIII-D Pelican access token, named `FDP_TOKEN`

#### Step 4 — Install ARGUS-FEDER

In [ ]:
import urllib.request
exec(urllib.request.urlopen(
    'https://raw.githubusercontent.com/klinucsd/argus_feder/main/install.py'
).read().decode(), globals())

## 1. Semantic concept → signal mapping 

**Purpose:** can ARGUS resolve a physics CONCEPT to the right MDSplus signal with
no symbol name given? Each request below deliberately omits the `\name`.
then short per-item lines, e.g.: 

### **Plasma current**

In [ ]:
%%ask
Fetch \ipmhd from efit01 for shot 165340 and plot it versus time, and
report the number of samples and the min/max with units.

### Elongation
**Ask:** "plasma elongation" (no `\kappa` given) → should resolve to `efit01/\kappa`.

In [ ]:
%%ask
For DIII-D shot 165340, fetch the plasma elongation from the MHD equilibrium
reconstruction, plot it versus time, and report its min and max. 

### Safety factor at 95% flux surface
**Ask:** no "MHD equilibrium" qualifier this time — the harder disambiguation case.  
**Expected:** resolves to `efit01/\q95` anyway.

In [ ]:
%%ask
For shot 165340, fetch the safety factor at the 95% flux surface, 
plot it versus time, and report its range. 

### Multi-shot overlay 
**Purpose:** test the fetch-many-shots-and-compare pattern, not just concept mapping.

In [ ]:
%%ask
For DIII-D shots 165340, 188702, and 194181, fetch the plasma current from the MHD
equilibrium reconstruction and overlay all three on a single chart. Report each
shot's peak current.

### Multi-shot shape comparison
**Purpose:** same multi-shot pattern, a dimensionless quantity (elongation) this time.

In [ ]:
%%ask
Compare the plasma shape across DIII-D shots 165340, 188702, and 194181 by fetching
the elongation from the MHD equilibrium reconstruction and overlaying it versus time
on one chart with a legend.

## 2. Metadata from Database

In [ ]:
%%ask
How many DIII-D plasma shots were there between 190000 and 195000?

In [ ]:
%%ask
What was shot 190000's elongation and peak plasma current?

In [ ]:
%%ask
When did DIII-D shot 165340 happen?

In [ ]:
%%ask
For DIII-D shot 144921, what fuel gas was used, what was the neutron yield,
and what was the peak normalized beta and when did it occur? 

In [ ]:
%%ask
Did DIII-D shot 96549 have an operational fault, and if so what kind?

In [ ]:
%%ask
What was the effective ion charge (Zeff) for DIII-D shot 165340?

In [ ]:
%%ask
What run date is DIII-D shot 165340 from, and who entered the record?

## 3. PTDATA — basic access test

**Purpose:** verify PTDATA (a separate data source from the MDSplus trees used
above) is reachable end to end through ARGUS. This was previously broken
(`error 22` / PTSERVER lookup failure) — fixed by installing the `fdp` package
and switching skill execution to `fdp run`.

**Ask:** fetch the plasma current from PTDATA (not efit01) for a known shot
and report basic stats.

**Expected:** a successful fetch, ~30,720 samples, peak current near 1.15 MA —
this is a raw digitizer trace, a different diagnostic pipeline than efit01's 
`\ipmhd`, so don't expect an identical peak to the efit01 numbers seen earlier
in this notebook.


In [ ]:
%%ask
Fetch the plasma current from PTDATA (not the MHD equilibrium reconstruction)
for DIII-D shot 165920, and report the number of samples and the min/max with
units.

In [ ]:
%%ask
For DIII-D shot 165920, fetch the plasma current from BOTH PTDATA and the
efit01 MHD equilibrium reconstruction, and compare their peak values and
units. Are they consistent with each other?

## 4. Signal meanings, paths, and IMAS names

Different ways to ask about the *same* signal -- plain-English meaning, its exact MDSplus path, and its IMAS (device-neutral) name -- to see how ARGUS handles each.

In [ ]:
%%ask
what is the meaning of \FS02UPDA and \EFIT01::BT0VAC in plain English?

In [ ]:
%%ask
What is the full path of \EFIT01::CPASMA? 

In [ ]:
%%ask
Do you know the IMAS name for \EFIT01::CPASMA? 

In [ ]:
%%ask
For DIII-D shot 165920, fetch equilibrium.time_slice[i].global_quantities.ip 
and plot it versus time